In [1]:
import os
import glob
import sys

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr3"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATH_FROM = f'../../data/lib2/{utr_type.upper()}_zinb_norm_2024-06-04.csv'
df_src = pd.read_csv(PATH_FROM)

In [6]:
df_src["seq"] = df_src["seq"].str.upper()

In [7]:
mc_data = df_src.groupby(["seq", "cell_type"])[["1", "2", "3", "4"]].sum().reset_index()
mc_data

,seq,cell_type,1,2,3,4
0,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c1,7803.0,7567.0,6317.0,6285.0
1,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c15,10545.0,10042.0,8972.0,10399.0
2,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c17,11655.0,10389.0,7803.0,6142.0
3,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c2,6274.0,5816.0,6332.0,6061.0
4,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c4,8586.0,7638.0,8481.0,5550.0
...,...,...,...,...,...,...
44750,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c15,257.0,423.0,434.0,265.0
44751,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c17,1391.0,2933.0,4367.0,3348.0
44752,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c2,150.0,74.0,29.0,9.0
44753,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c4,6677.0,4461.0,3281.0,2592.0


In [8]:
mc_data["mass_center"] = (mc_data[["1", "2", "3", "4"]] * np.arange(1, 5)).sum(axis=1) / mc_data[["1", "2", "3", "4"]].sum(axis=1)

In [9]:
cell_types = ["c1", "c13", "c17", "c2", "c4", "c6"]
num_classes = len(cell_types)

In [10]:
from itertools import product

df = pd.DataFrame(product(df_src["seq"].unique(), cell_types), columns=["seq", "cell_type"])
df.head()

,seq,cell_type
0,CATCAGAAGCTGCTTGTGTATGTAAGGAAAATGGGGCTTCCTCCTA...,c1
1,CATCAGAAGCTGCTTGTGTATGTAAGGAAAATGGGGCTTCCTCCTA...,c13
2,CATCAGAAGCTGCTTGTGTATGTAAGGAAAATGGGGCTTCCTCCTA...,c17
3,CATCAGAAGCTGCTTGTGTATGTAAGGAAAATGGGGCTTCCTCCTA...,c2
4,CATCAGAAGCTGCTTGTGTATGTAAGGAAAATGGGGCTTCCTCCTA...,c4


In [11]:
df = pd.merge(df, mc_data[["seq", "cell_type", "mass_center"]], on=["seq", "cell_type"], how="left", validate="1:1")

In [12]:
batch_size = 1024

In [13]:
num_workers = 32

In [14]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [15]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [16]:
ckpt = torch.load(MODEL_PATH)

In [17]:
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding
ckpt["hyper_parameters"]["model_class"]

legnet_embedding.LegNetEmbedding

In [18]:
loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

<All keys matched successfully>

In [19]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[0],
    deterministic=True,
    # gradient_clip_val=1e-5,
    # gradient_clip_algorithm="norm",
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [20]:
pred_tuples, _  = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [21]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")
seq_embedding

array([[ 0.05054361,  0.13758536, -0.00781247, ...,  0.1780348 ,
         0.1417292 ,  0.17395762],
       [ 0.01212397,  0.1102045 , -0.02112725, ...,  0.26047695,
         0.16057268,  0.16104607],
       [ 0.07559448,  0.12510045,  0.09698768, ...,  0.16845468,
         0.11722579,  0.15285757],
       ...,
       [-0.01973554,  0.0601767 ,  0.08728627, ..., -0.08754272,
         0.01245509,  0.10119739],
       [ 0.15837865,  0.177551  ,  0.1064001 , ...,  0.16573565,
         0.11471251,  0.16896105],
       [ 0.18903595,  0.1558206 ,  0.05220386, ...,  0.21771356,
         0.16245542,  0.24547835]], dtype=float32)

In [22]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)
pred_mass_center

array([[2.4477606, 2.6312623, 2.71796  , 2.7015514, 2.6644154, 2.652275 ],
       [2.3836303, 2.6044245, 2.692752 , 2.6899467, 2.64538  , 2.7086234],
       [2.4330878, 2.5544274, 2.4583511, 2.4820342, 2.4567616, 2.4892306],
       ...,
       [2.0895789, 2.6110866, 2.6628492, 2.4380877, 2.4290364, 1.9720131],
       [2.9206474, 2.6365292, 2.855594 , 2.7389164, 2.7933319, 2.8830488],
       [2.8910835, 2.721988 , 3.2020404, 2.9371183, 2.8878555, 3.0686786]],
      dtype=float32)

In [23]:
result = df.pivot(columns="cell_type", index="seq", values="mass_center")
result.columns = pd.MultiIndex.from_product([["mass_center"], result.columns])
result.head()

mass_center                \
cell_type                                                   c1 c13       c17   
seq                                                                            
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...    2.396253 NaN  2.234294   
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...    2.406223 NaN  2.227175   
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC...    2.121921 NaN  1.973587   
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...    2.574347 NaN  2.485379   
AAAAAAAAAAAAAAAAAAAACATTAAACTGCCTAACTCCAATTTTGT...    2.946025 NaN       NaN   

                                                                        \
cell_type                                                 c2        c4   
seq                                                                      
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...  2.497488  2.363411   
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...  2.514426  2.374429   
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC...  2.180578  2.216967   
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...  2.468296  2.293195   
AAAAAAAAAAAAAAAAAAAACATTAAACTGCCTAACTCCAATTTTGT...  3.048159       NaN   

                                                              
cell_type                                                 c6  
seq                                                           
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...  2.415970  
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...  2.467038  
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC...  2.448059  
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...  2.473553  
AAAAAAAAAAAAAAAAAAAACATTAAACTGCCTAACTCCAATTTTGT...  2.460606

In [24]:
result[
    pd.MultiIndex.from_product([["pred_mass_center"], result["mass_center"].columns])
] = pred_mass_center
result["pred_mass_center"].head()

cell_type,c1,c13,c17,c2,c4,c6
seq,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTCCCCCCCCCCCCCCCCCCCCATCCCCCCTCCCCTCCTTCCTCCCCCCCCCCACCCCCTTCCTTTTCCCCCTTCTTATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTG,2.447761,2.631262,2.717960,2.701551,2.664415,2.652275
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTCCCCCCCCCTCCCCCTCCCCTCCCCCCCCCACCCCCCTCTCCCCACCTCCTCGCCCCCTCTCCCCCTAGCCCCCTCCAGCACCCAGCACATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTG,2.383630,2.604424,2.692752,2.689947,2.645380,2.708623
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCCCAGCTCAATGGGAGGTTCCCAACACTACTCCTTCTCTCTTCTCTTTCCTGCCTCCTTTCTCTCATGTGAAAAGCAGTGTGGGTGTAGCCCTCTCAAACAGAGACAGAGGGGGGAGGGAAGTGCCATTCAGAGAGGGGGAGGGCAAGCTGTGCACACAACAGTGATTCTGATCCCTGCTCTTTCCCTTCCCAGC,2.433088,2.554427,2.458351,2.482034,2.456762,2.489231
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCCCCCCATCCCACCCCTCCCCCTCCCCCCCCCACCCCCCACCCCCTCACCTGTCCTCCATCACACGGTCACTTCCTGTTAAAAACACTTGCCTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTATATATATGG,2.905467,2.669903,3.094280,2.914595,3.023996,2.967431
AAAAAAAAAAAAAAAAAAAACATTAAACTGCCTAACTCCAATTTTGTCAGTTGCTCTGCTTTTTTCATAGTAAAGCTTTTCCCTGAATTTCAGTGATGGAATCATTTTATCATATGTTTCTGATAAGAGATGAATCTGTGTCTTATCCTTACCTAGCTAGCTGCTTTGCTGCATGTGTATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACTAAATATTTGGCTTACACTAATCCCA,2.871949,2.719429,3.240425,3.004111,3.137769,3.018033


In [25]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [26]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c6_022,c6_023,c6_024,c6_025,c6_026,c6_027,c6_028,c6_029,c6_030,c6_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTCCCCCCCCCCCCCCCCCCCCATCCCCCCTCCCCTCCTTCCTCCCCCCCCCCACCCCCTTCCTTTTCCCCCTTCTTATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTG,0.050544,0.137585,-0.007812,0.075960,0.051088,0.240602,-0.034394,0.017846,0.247296,0.142552,...,0.044018,0.052009,0.035774,0.139195,0.132614,0.168359,0.068760,0.178035,0.141729,0.173958
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTCCCCCCCCCTCCCCCTCCCCTCCCCCCCCCACCCCCCTCTCCCCACCTCCTCGCCCCCTCTCCCCCTAGCCCCCTCCAGCACCCAGCACATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTG,0.012124,0.110204,-0.021127,0.070172,0.058500,0.288753,-0.075980,-0.003226,0.256835,0.176046,...,0.068889,0.040567,0.045914,0.131437,0.153443,0.177389,0.064070,0.260477,0.160573,0.161046
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCCCAGCTCAATGGGAGGTTCCCAACACTACTCCTTCTCTCTTCTCTTTCCTGCCTCCTTTCTCTCATGTGAAAAGCAGTGTGGGTGTAGCCCTCTCAAACAGAGACAGAGGGGGGAGGGAAGTGCCATTCAGAGAGGGGGAGGGCAAGCTGTGCACACAACAGTGATTCTGATCCCTGCTCTTTCCCTTCCCAGC,0.075594,0.125100,0.096988,0.134730,0.102105,0.064863,0.034174,0.175896,0.100989,0.093835,...,0.096221,0.089477,0.077765,0.107992,0.119779,0.164694,0.057285,0.168455,0.117226,0.152858
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCCCCCCATCCCACCCCTCCCCCTCCCCCCCCCACCCCCCACCCCCTCACCTGTCCTCCATCACACGGTCACTTCCTGTTAAAAACACTTGCCTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTATATATATGG,0.104266,0.191629,-0.019856,0.062690,0.027561,0.074583,0.156849,0.149849,0.174217,0.085869,...,0.077108,0.015663,0.032370,0.164948,0.169224,0.218003,0.110508,0.242311,0.204629,0.199482
AAAAAAAAAAAAAAAAAAAACATTAAACTGCCTAACTCCAATTTTGTCAGTTGCTCTGCTTTTTTCATAGTAAAGCTTTTCCCTGAATTTCAGTGATGGAATCATTTTATCATATGTTTCTGATAAGAGATGAATCTGTGTCTTATCCTTACCTAGCTAGCTGCTTTGCTGCATGTGTATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACTAAATATTTGGCTTACACTAATCCCA,0.055881,0.168320,-0.020338,0.027427,0.035002,0.148324,0.163176,0.105954,0.181225,0.087004,...,0.104142,0.060090,0.032568,0.150439,0.154687,0.189236,0.125419,0.227003,0.228939,0.177916
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTTTTAAGTTTTTTGTAAAATTGCCTTATGCCACTGCATCTGGAAAACTGATGGCTGCCACAGTTGACATGTGTAGCTTATGAATATAAATGCTATTTTCAGCTTTTTATAAAATTATAGTAGGCATGCTAGGAAGCATAGTGTAATAAATAATCCTTTTGCTTGTAATGGATAGTTTAGTGTTACTTTATGCATGGTGCTGTCTAAACTGAGCCAGTTGGAGCATAAGCTGGGG,0.069787,0.150878,0.036832,0.109331,0.063502,0.061301,0.213007,0.217073,0.017458,0.043834,...,0.092022,0.076691,0.034434,0.091172,0.091340,0.088425,0.129085,0.049989,0.151966,0.123528
TTTTTTTTTTTATATATTTTTTTTTTTTATAAAATTTATTTTTTTAATTTTTTTTTAAAAAAAAAAAAAAAAAAAAAAAATGCTGATTGAGGCCGAGGGTGGTGAGGGTGGGGGCGGTGAGCCTGTGGGGCTGAGTGTGCCCAAAGTCGGCTGTCCCCCAGCCCCGGGCCTCCGTGGGCAGCAGTGACTTGAGGCTTGACCTCCCCAGCCCCCCACCCCCACCTGACCCCACCGGCCCCC,0.127709,0.175382,-0.020531,0.130600,-0.016901,-0.075038,0.611208,0.235334,-0.006559,-0.019946,...,0.086054,0.139703,0.001738,0.060227,0.060799,0.042139,0.174689,-0.043209,0.187247,0.072212
TTTTTTTTTTTTAAATTTAAAATATTTTTTTGATTGGAGCAAAACAGATCAGAAGGTGGCATGTGCAGCCCATCCTGCCCCTGTCCAGTCAGTTGTGATGGGGCCTGAGGTAGGTTGAGTGCCTCCATGTCTGGCAGAACAAGGAGGGTCACAGAAGTTCAAGCACTAGCCTGACCACGGTCCTGTCTGTGGGGAGCGTGAGGCCGCTGCCCAACCTCACTCCCCTCCAGCTCATGTGCC,-0.019736,0.060177,0.087286,0.141929,0.169348,0.305569,-0.110322,0.172734,0.063541,0.111523,...,0.058489,0.088426,0.090783,0.046591,0.076761,-0.000209,0.077846,-0.087543,0.012455,0.101197


### Counting k-mers

In [27]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [28]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [29]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTCCCCCCCCCCCCCCCCCCCCATCCCCCCTCCCCTCCTTCCTCCCCCCCCCCACCCCCTTCCTTTTCCCCCTTCTTATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTG,36,66,59,79,26,6,0,4,7,50,...,0,1,0,0,0,58,1,5,0,2
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTCCCCCCCCCTCCCCCTCCCCTCCCCCCCCCACCCCCCTCTCCCCACCTCCTCGCCCCCTCTCCCCCTAGCCCCCTCCAGCACCCAGCACATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTG,35,78,58,69,25,5,3,2,8,54,...,1,2,0,1,0,52,0,0,0,0
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCCCAGCTCAATGGGAGGTTCCCAACACTACTCCTTCTCTCTTCTCTTTCCTGCCTCCTTTCTCTCATGTGAAAAGCAGTGTGGGTGTAGCCCTCTCAAACAGAGACAGAGGGGGGAGGGAAGTGCCATTCAGAGAGGGGGAGGGCAAGCTGTGCACACAACAGTGATTCTGATCCCTGCTCTTTCCCTTCCCAGC,66,64,51,59,33,8,18,7,18,19,...,0,13,3,4,2,6,0,11,0,4
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCCCCCCATCCCACCCCTCCCCCTCCCCCCCCCACCCCCCACCCCCTCACCTGTCCTCCATCACACGGTCACTTCCTGTTAAAAACACTTGCCTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTATATATATGG,46,72,53,69,26,12,0,8,13,49,...,0,0,0,1,1,48,1,1,1,0
AAAAAAAAAAAAAAAAAAAACATTAAACTGCCTAACTCCAATTTTGTCAGTTGCTCTGCTTTTTTCATAGTAAAGCTTTTCCCTGAATTTCAGTGATGGAATCATTTTATCATATGTTTCTGATAAGAGATGAATCTGTGTCTTATCCTTACCTAGCTAGCTGCTTTGCTGCATGTGTATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACTAAATATTTGGCTTACACTAATCCCA,67,39,44,90,32,7,8,19,11,8,...,0,4,4,7,2,21,5,4,4,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTTTTAAGTTTTTTGTAAAATTGCCTTATGCCACTGCATCTGGAAAACTGATGGCTGCCACAGTTGACATGTGTAGCTTATGAATATAAATGCTATTTTCAGCTTTTTATAAAATTATAGTAGGCATGCTAGGAAGCATAGTGTAATAAATAATCCTTTTGCTTGTAATGGATAGTTTAGTGTTACTTTATGCATGGTGCTGTCTAAACTGAGCCAGTTGGAGCATAAGCTGGGG,66,32,51,91,22,6,15,23,11,5,...,0,2,4,9,6,7,8,1,6,21
TTTTTTTTTTTATATATTTTTTTTTTTTATAAAATTTATTTTTTTAATTTTTTTTTAAAAAAAAAAAAAAAAAAAAAAAATGCTGATTGAGGCCGAGGGTGGTGAGGGTGGGGGCGGTGAGCCTGTGGGGCTGAGTGTGCCCAAAGTCGGCTGTCCCCCAGCCCCGGGCCTCCGTGGGCAGCAGTGACTTGAGGCTTGACCTCCCCAGCCCCCCACCCCCACCTGACCCCACCGGCCCCC,55,59,56,70,29,6,11,9,8,35,...,1,0,9,2,4,3,5,0,3,32
TTTTTTTTTTTTAAATTTAAAATATTTTTTTGATTGGAGCAAAACAGATCAGAAGGTGGCATGTGCAGCCCATCCTGCCCCTGTCCAGTCAGTTGTGATGGGGCCTGAGGTAGGTTGAGTGCCTCCATGTCTGGCAGAACAAGGAGGGTCACAGAAGTTCAAGCACTAGCCTGACCACGGTCCTGTCTGTGGGGAGCGTGAGGCCGCTGCCCAACCTCACTCCCCTCCAGCTCATGTGCC,52,60,63,65,14,8,20,10,20,22,...,0,2,6,5,5,7,2,1,4,16


In [30]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_library2.parquet")

---